9 - Catalogo de Dados

In [0]:
from pyspark.sql import functions as F

# ============================================================
# TABELAS AVALIADAS — APENAS CAMADA SILVER 
# ============================================================
tables = [
    "silver_full",
    "silver_gols",
    "silver_cartoes",
    "silver_estatisticas"
]

catalog = []

for table in tables:
    try:
        df = spark.table(table)
        row_count = df.count()
    except Exception as e:
        print(f"Erro ao acessar tabela {table}: {e}")
        continue

    for field in df.schema.fields:
        col_name = field.name
        col_type = field.dataType.simpleString()

        # ============================================================
        # DOMÍNIO — até 10 valores distintos
        # ============================================================
        try:
            dominio_values = (
                df.select(F.col(col_name).cast("string").alias("val"))
                  .dropna()
                  .distinct()
                  .limit(10)
                  .collect()
            )
            dominio_str = ", ".join([row["val"] for row in dominio_values]) or "N/A"
        except:
            dominio_str = "N/A"

        # ============================================================
        # VALORES MÍNIMOS E MÁXIMOS (apenas numéricos)
        # ============================================================
        if col_type in ["int", "bigint", "double", "float", "long", "decimal"]:
            try:
                min_val = df.select(F.min(col_name)).first()[0]
                max_val = df.select(F.max(col_name)).first()[0]
                min_val = str(min_val) if min_val is not None else "NULL"
                max_val = str(max_val) if max_val is not None else "NULL"
            except:
                min_val, max_val = "NULL", "NULL"
        else:
            min_val = "N/A"
            max_val = "N/A"

        # ============================================================
        # DESCRIÇÃO TÉCNICA
        # ============================================================
        descricao = (
            f"Coluna '{col_name}' da tabela '{table}', do tipo '{col_type}'. "
            f"Representa atributo estruturado da camada Silver, utilizado para "
            f"validação, padronização e posterior modelagem dimensional."
        )

        catalog.append((table, col_name, col_type, dominio_str, min_val, max_val, descricao))

# ============================================================
# DATAFRAME FINAL DO CATÁLOGO
# ============================================================
catalog_df = spark.createDataFrame(
    catalog,
    ["tabela", "coluna", "tipo", "dominio_exemplo", "minimo", "maximo", "descricao"]
)

display(catalog_df.orderBy("tabela", "coluna"))


tabela,coluna,tipo,dominio_exemplo,minimo,maximo,descricao
silver_cartoes,atleta,string,"Paulo Roberto da Silva, Thiago Heleno, Andrés D'Alessandro, Marcelo Machado dos Santos, Mansur, Juan Silveira dos Santos, Vinícius Santos Silva, Carlos Emiliano Pereira, Cristian Chagas Tarouco, Rhayner",N/A,N/A,"Coluna 'atleta' da tabela 'silver_cartoes', do tipo 'string'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,cartao,string,"Amarelo, Vermelho",N/A,N/A,"Coluna 'cartao' da tabela 'silver_cartoes', do tipo 'string'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,clube,string,"Figueirense, Internacional, Vitoria, Coritiba, Bahia, Cruzeiro, Botafogo-rj, Sao Paulo, Athletico-pr, Gremio",N/A,N/A,"Coluna 'clube' da tabela 'silver_cartoes', do tipo 'string'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,ingestao_dt,timestamp,2025-12-15 22:55:38.119914,N/A,N/A,"Coluna 'ingestao_dt' da tabela 'silver_cartoes', do tipo 'timestamp'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,minuto,string,"66, 44, 72, 86, 10, 20, 82, 76, 78, 41",N/A,N/A,"Coluna 'minuto' da tabela 'silver_cartoes', do tipo 'string'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,minuto_int,int,"66, 44, 72, 86, 10, 20, 82, 76, 78, 41",0,108,"Coluna 'minuto_int' da tabela 'silver_cartoes', do tipo 'int'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,num_camisa,int,"28, 4, 10, 29, 30, 3, 15, 9, 40, 7",1,700,"Coluna 'num_camisa' da tabela 'silver_cartoes', do tipo 'int'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,partida_id,int,"4607, 4608, 4609, 4612, 4611, 4610, 4614, 4615, 4616, 4617",4607,8405,"Coluna 'partida_id' da tabela 'silver_cartoes', do tipo 'int'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,posicao,string,"Zagueiro, Meio-campo, Atacante, Goleiro",N/A,N/A,"Coluna 'posicao' da tabela 'silver_cartoes', do tipo 'string'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
silver_cartoes,rodata,int,"1, 2, 3, 4, 5, 6, 7, 8, 9, 28",1,38,"Coluna 'rodata' da tabela 'silver_cartoes', do tipo 'int'. Representa atributo estruturado da camada Silver, utilizado para validação, padronização e posterior modelagem dimensional."
